# Deep Learning — BiLSTM + Attention (PyTorch)

**Arsitektur:** Embedding(30k×64) → BiLSTM(128) → BiLSTM(128) → Attention → FC(256→64→2)
**Dijalankan di:** Kaggle Notebook dengan GPU

**Hyperparameter** dari Optuna Trial #19 (Val Loss: 0.0886 | Val Acc: 0.9676)

**Workflow:**
1. Jalankan `02_preprocessing.ipynb` lokal → hasilkan `cleaned_text.csv`
2. Upload `cleaned_text.csv` sebagai Kaggle Dataset
3. Jalankan notebook ini di Kaggle dengan GPU
4. Download output dari `/kaggle/working/` → simpan di `models/model_dl/`


## Environment Detection


In [ ]:
import os
IS_KAGGLE = os.path.exists('/kaggle')
DATA_PATH  = '/kaggle/input/sentiment-cleaned/cleaned_text.csv' if IS_KAGGLE else '../data/processed/cleaned_text.csv'
OUTPUT_DIR = '/kaggle/working/' if IS_KAGGLE else '../models/model_dl/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Environment : {"Kaggle" if IS_KAGGLE else "Lokal"}')
print(f'DATA_PATH   : {DATA_PATH}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')

## 1. Import Library


In [ ]:
from __future__ import annotations
import json, logging, time
from collections import Counter
from dataclasses import dataclass
from typing import List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

logging.basicConfig(level=logging.INFO,
    format='[%(levelname)s] %(asctime)s — %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')

## 2. Konfigurasi — Nilai Optimal Menggunakan Algoritma Optuna

| Parameter | Nilai |
|---|---|
| `lr` | **0.003814** |
| `dropout` | **0.4483** |
| `weight_decay` | **0.001070** |
| `label_smoothing` | **7.8e-5** |
| `embed_dim` | **64** |
| `hidden_dim1` | **128** |
| `hidden_dim2` | **128** |
| `batch_size` | **128** |


In [ ]:
@dataclass
class DLConfig:
    data_path:        str   = DATA_PATH
    model_save_dir:   str   = OUTPUT_DIR
    lr:               float = 0.003814      
    dropout:          float = 0.4483        
    weight_decay:     float = 0.001070      
    label_smoothing:  float = 7.805e-05     
    embed_dim:        int   = 64            
    hidden_dim1:      int   = 128           
    hidden_dim2:      int   = 128           
    batch_size:       int   = 128           
    # --- Fixed params ---
    vocab_size:       int   = 30_000
    max_len:          int   = 128
    num_classes:      int   = 2
    num_epochs:       int   = 30            # lebih banyak epoch karena LR lebih besar
    patience:         int   = 5
    test_size:        float = 0.2
    seed:             int   = 42

cfg = DLConfig()
print(cfg)

## 3. Class Definitions


In [ ]:
class Vocabulary:
    PAD_IDX, UNK_IDX = 0, 1
    def __init__(self, max_size=30_000):
        self.max_size = max_size
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
    def build(self, texts):
        counter = Counter()
        for t in texts: counter.update(str(t).split())
        logger.info('Total token unik: %d', len(counter))
        for word, _ in counter.most_common(self.max_size - 2):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
        logger.info('Vocabulary: %d token', len(self.word2idx))
        return self
    def encode(self, text, max_len=128):
        tokens = str(text).split()[:max_len]
        ids = [self.word2idx.get(t, self.UNK_IDX) for t in tokens]
        ids += [self.PAD_IDX] * (max_len - len(ids))
        return ids
    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(self.word2idx, f, ensure_ascii=False)
    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            word2idx = json.load(f)
        v = cls(max_size=len(word2idx))
        v.word2idx = word2idx
        v.idx2word = {int(v2): k for k, v2 in word2idx.items()}
        return v
    def __len__(self): return len(self.word2idx)

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.X = torch.tensor([vocab.encode(t, max_len) for t in texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

In [ ]:
class BiLSTMAttention(nn.Module):
    """Embedding(64) → BiLSTM(128) → BiLSTM(128) → Attention → FC(256→64→2)"""
    def __init__(self, vocab_size, embed_dim=64, hidden_dim1=128,
                 hidden_dim2=128, num_classes=2, dropout=0.4483):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(embed_dim,     hidden_dim1, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden_dim1*2, hidden_dim2, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(hidden_dim2*2, 1)
        self.dropout   = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim2*2, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        emb    = self.dropout(self.embedding(x))
        out1,_ = self.lstm1(emb);  out1 = self.dropout(out1)
        out2,_ = self.lstm2(out1); out2 = self.dropout(out2)
        attn_w = torch.softmax(self.attention(out2), dim=1)
        ctx    = (attn_w * out2).sum(dim=1)
        return self.fc(self.dropout(ctx))

## 4. Fungsi Utilitas


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total   += len(y)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total   += len(y)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return total_loss/total, correct/total, np.array(all_preds), np.array(all_labels)

---
## Pipeline Training


### Step 1 — Load Data


In [ ]:
set_seed(cfg.seed)
df = pd.read_csv(cfg.data_path)
df['clean_text'] = df['clean_text'].fillna('').astype(str)
texts  = df['clean_text'].tolist()
labels = df['sentiment'].tolist()
print(f'✅ Data dimuat: {len(df)} sampel')
print(f'   Distribusi: {dict(Counter(labels))}')
df.head(3)

### Step 2 — Build Vocabulary


In [ ]:
vocab = Vocabulary(max_size=cfg.vocab_size).build(texts)
vocab.save(os.path.join(cfg.model_save_dir, 'vocab_dl.json'))
print(f'✅ Vocabulary: {len(vocab)} token')

### Step 3 — Split & DataLoader


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=cfg.test_size, random_state=cfg.seed, stratify=labels
)
print(f'Train: {len(X_train)} | Val: {len(X_val)}')
train_ds = SentimentDataset(X_train, y_train, vocab, cfg.max_len)
val_ds   = SentimentDataset(X_val,   y_val,   vocab, cfg.max_len)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,   shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size*2, shuffle=False, num_workers=0, pin_memory=True)

### Step 4 — Build Model


In [ ]:
model = BiLSTMAttention(
    vocab_size  = len(vocab),
    embed_dim   = cfg.embed_dim,
    hidden_dim1 = cfg.hidden_dim1,
    hidden_dim2 = cfg.hidden_dim2,
    num_classes = cfg.num_classes,
    dropout     = cfg.dropout
).to(DEVICE)
n_params = count_parameters(model)
print(f'✅ Model: BiLSTM + Attention | Parameter: {n_params:,}')
print(model)

### Step 5 — Training Loop


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

best_val_acc, best_val_f1 = 0.0, 0.0
best_val_loss = float('inf')
patience_counter = 0
history = []
prev_lr = cfg.lr

print(f'{"Epoch":<6} {"TrainLoss":<12} {"TrainAcc":<12} {"ValLoss":<12} {"ValAcc":<12} {"LR":<12}')
print('-' * 74)

for epoch in range(1, cfg.num_epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_preds, vl_labels = evaluate(model, val_loader, criterion)
    vl_f1 = f1_score(vl_labels, vl_preds, average='weighted')

    scheduler.step(vl_loss)
    lr_now = optimizer.param_groups[0]['lr']
    if lr_now < prev_lr:
        print(f'  ⬇️  LR: {prev_lr:.6f} → {lr_now:.6f}')
    prev_lr = lr_now

    print(f'{epoch:<6} {tr_loss:<12.4f} {tr_acc:<12.4f} {vl_loss:<12.4f} {vl_acc:<12.4f} {lr_now:<12.6f} ({time.time()-t0:.1f}s)')

    history.append({'epoch': epoch, 'train_loss': round(tr_loss,4), 'train_acc': round(tr_acc,4),
                    'val_loss': round(vl_loss,4), 'val_acc': round(vl_acc,4), 'val_f1': round(vl_f1,4)})

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_val_acc  = vl_acc
        best_val_f1   = vl_f1
        torch.save(model.state_dict(), os.path.join(cfg.model_save_dir, 'bilstm_attention.pt'))
        print(f'  ✅ Best model saved (val_loss={best_val_loss:.4f} | val_acc={best_val_acc:.4f})')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f'  Early stopping setelah {epoch} epoch.')
            break

print(f'\n✅ Training selesai | Best Val Loss: {best_val_loss:.4f} | Val Acc: {best_val_acc:.4f} | Val F1: {best_val_f1:.4f}')

### Step 6 — Plot Kurva


In [ ]:
epochs_ran = [h['epoch']      for h in history]
tr_losses  = [h['train_loss'] for h in history]
vl_losses  = [h['val_loss']   for h in history]
tr_accs    = [h['train_acc']  for h in history]
vl_accs    = [h['val_acc']    for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_ran, tr_losses, 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_ran, vl_losses, 'r-o', label='Val Loss',   linewidth=2)
axes[0].set_title('Training vs Validation Loss', fontsize=14)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, tr_accs, 'b-o', label='Train Acc', linewidth=2)
axes[1].plot(epochs_ran, vl_accs, 'r-o', label='Val Acc',   linewidth=2)
axes[1].set_title('Training vs Validation Accuracy', fontsize=14)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.model_save_dir, 'training_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Kurva tersimpan:', os.path.join(cfg.model_save_dir, 'training_curve.png'))

### Step 7 — Evaluasi Final


In [ ]:
model.load_state_dict(
    torch.load(os.path.join(cfg.model_save_dir, 'bilstm_attention.pt'), map_location=DEVICE)
)
_, final_acc, final_preds, final_labels = evaluate(model, val_loader, criterion)
final_f1 = f1_score(final_labels, final_preds, average='weighted')

print('=' * 55)
print('  HASIL EVALUASI FINAL')
print('=' * 55)
print(f'  Accuracy     : {final_acc:.4f}')
print(f'  F1 (weighted): {final_f1:.4f}')
print()
print(classification_report(final_labels, final_preds,
      target_names=['Negatif (0)', 'Positif (1)'], digits=4))
print('Confusion Matrix:')
print(confusion_matrix(final_labels, final_preds))
print('=' * 55)

### Step 8 — Simpan Artefak


In [ ]:
history_path = os.path.join(cfg.model_save_dir, 'bilstm_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

config_path = os.path.join(cfg.model_save_dir, 'bilstm_config.json')
with open(config_path, 'w') as f:
    json.dump(cfg.__dict__, f, indent=2)

print('✅ Artefak tersimpan:')
print(f'   Model   : {os.path.join(cfg.model_save_dir, "bilstm_attention.pt")}')
print(f'   Vocab   : {os.path.join(cfg.model_save_dir, "vocab_dl.json")}')
print(f'   History : {history_path}')
print(f'   Config  : {config_path}')
print(f'   Kurva   : {os.path.join(cfg.model_save_dir, "training_curve.png")}')
if IS_KAGGLE:
    print('\n📥 Download semua file dari /kaggle/working/')
    print('   lalu letakkan di models/model_dl/ di lokal.')